# AmsterdamUMCdb Cohort Construction

## Objectives

*   Replicate [the Zappalà et al. (2025)](https://www.sciencedirect.com/science/article/pii/S0883944125000929) cohort definition (adult, Invasive Mechanical Ventillation > 24h, ECMO excluded.
*   Derive ventilation concepts from the OMOP vocabulary hierarchy and validate against data.
*   Construct ventilation episoded using the 48-hour gap rule.
*   Apply inclusion/exclusion filters and resolve repeat admissions
*   Validate the cohort against the paper's size and age/sex distributions.






AI use declaration. Claude (Claude Opus 4.8, Anthropic) assisted with drafting and debugging code and suggesting validation approaches. All queries were run and verified by me, and all methodological decisions are my own.

![AmsterdamUMCdb OMOP Schema](https://raw.githubusercontent.com/HatolkarAV/ICU-Weaning-Prediction/main/img/amsterdamumcdb_omop_schema_attributed.png)

AI-assisted (Claude, Anthropic — Claude Opus 4.8): schema diagram of relevant tables generated with AI from [AmsterdamUMCdb OMOP v5.4 docs](https://ohdsi.github.io/CommonDataModel/cdm54.html).*

This notebook replicates the AmsterdamUMCdb cohort construction from Zappalà et al. (2025), which developed an ML model for predicting invasive mechanical ventilation (IMV) weaning readiness.
[Development and validation of a machine learning model for real-time prediction of invasive mechanical ventilation weaning readiness](https://www.sciencedirect.com/science/article/pii/S0883944125000929)

### Imports and configuration

**Initial BigQuery Setup by Ayushi Kashyap - Adapted for this notebook.**

In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt

In [ ]:
PROJECT_ID         = 'capstoneweaningprediction' #@param {type:"string"}
DATASET_PROJECT_ID = 'amsterdamumcdb'            #@param {type:"string"}
DATASET_ID         = 'version1_5_0'              #@param {type:"string"}
LOCATION           = 'eu'                        #@param {type:"string"}

In [ ]:
import os
from google.colab import auth
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
auth.authenticate_user()
print('Authenticated')

In [ ]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable
DataTable.max_columns = 50
DataTable.max_rows    = 80000

In [ ]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
def_config = bigquery.job.QueryJobConfig(
    default_dataset=DATASET_PROJECT_ID + '.' + DATASET_ID
)
bigquery_magics.context.default_query_job_config = def_config
client = bigquery.Client(
    project=PROJECT_ID, location=LOCATION,
    default_query_job_config=def_config
)
print('BigQuery client ready')

### Invasive Mechanical Ventillation concepts with count

AI-assisted: hierarchy-validation approach suggested by Claude, Anthropic — Claude Opus 4.8

In [ ]:
query = f"""
WITH seed AS (
    -- Seed Concepts: ventilation
    SELECT concept_id
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
    WHERE standard_concept = 'S'
      AND domain_id = 'Procedure'
      AND LOWER(concept_name) LIKE '%ventilation%'
),
family AS (
    -- Family of seed concepts: ventilation
    SELECT DISTINCT ca.descendant_concept_id AS concept_id
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept_ancestor` ca
    JOIN seed s ON ca.ancestor_concept_id = s.concept_id
    UNION DISTINCT
    SELECT concept_id FROM seed
)
SELECT
    c.concept_id,
    c.concept_name,
    COUNT(DISTINCT po.visit_occurrence_id) AS visit_count
FROM family f
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
    ON c.concept_id = f.concept_id
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence` po
    ON po.procedure_concept_id = c.concept_id
   AND po.provider_id IS NOT NULL
WHERE c.domain_id = 'Procedure'
  -- Clinical exclusions: non-invasive / non-mechanical modes are not IMV
  AND LOWER(c.concept_name) NOT LIKE '%non-invasive%'
  AND LOWER(c.concept_name) NOT LIKE '%noninvasive%'
  AND LOWER(c.concept_name) NOT LIKE '%continuous positive airway pressure%'
  AND LOWER(c.concept_name) NOT LIKE '%bilevel positive airway pressure%'
  AND LOWER(concept_name) NOT LIKE '%positive airway pressure%'
GROUP BY c.concept_id, c.concept_name
HAVING visit_count > 0
ORDER BY visit_count DESC
"""
imv_concepts = client.query(query).to_dataframe()
IMV_CONCEPTS_SQL = ','.join(str(c) for c in imv_concepts['concept_id'].tolist())

print(f'IMV concepts (derived from seed + hierarchy): {len(imv_concepts)}')
imv_concepts

Three modes dominate (Pressure Support and the two Continuous Mandatory modes = 34,850. These are standard ICU methods.

### Pull ICU visits and patients demographics

AI-assisted: query reviewed with Claude, Anthropic — Claude Opus 4.8; join integrity and demographics.

[OHDSI BLOG](https://forums.ohdsi.org/t/standard-values-for-gender/15614) reference link for gender labels

In [ ]:
query = f"""
SELECT
    vo.person_id,
    vo.visit_occurrence_id,
    vo.visit_start_datetime,
    vo.visit_end_datetime,
    vo.preceding_visit_occurrence_id,
    p.year_of_birth,
    p.gender_concept_id,
    EXTRACT(YEAR FROM vo.visit_start_date)                    AS admission_year,
    EXTRACT(YEAR FROM vo.visit_start_date) - p.year_of_birth  AS age_at_admission
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence` vo
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.person` p
    ON vo.person_id = p.person_id
ORDER BY 3 ASC
"""
visits = client.query(query).to_dataframe()
visits['visit_occurrence_id'] = visits['visit_occurrence_id'].astype('int64')

# Map gender to readable labels
gender_map = {8507: 'Male', 8532: 'Female'}
visits['gender_label'] = visits['gender_concept_id'].map(gender_map).fillna('Unknown')

# Confirm the join behaved as validated
assert len(visits) == visits['visit_occurrence_id'].nunique(), "Duplicate visit rows"
assert visits['year_of_birth'].notna().all(), "Null birth year present — would produce NaN ages"

print(f'Loaded {len(visits):,} visits, {visits["person_id"].nunique():,} unique patients')
print(visits['gender_label'].value_counts())
visits.head()

Extracted all ICU admissions from the database, about 23106 visits from the 20109 patients tagged with patient's age and gender. The patient count is lower than visit count means some patient admitted multiple times.

### Filter the ECMO patient visits (Extracorporeal Membrane Oxygenation)

AI-assisted (Claude, Anthropic — Claude Opus 4.8): ECMO source-value query drafted with AI

In [ ]:
## Identify ECMO visits (exclusion)

query = f"""
SELECT DISTINCT visit_occurrence_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence`
WHERE LOWER(procedure_source_value) LIKE '%ecmo%'
  AND provider_id IS NOT NULL
UNION DISTINCT
SELECT DISTINCT visit_occurrence_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.device_exposure`
WHERE LOWER(device_source_value) LIKE '%ecmo%'
  AND provider_id IS NOT NULL
UNION DISTINCT
SELECT DISTINCT visit_occurrence_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
WHERE LOWER(measurement_source_value) LIKE '%ecmo%'
  AND provider_id IS NOT NULL
"""
ecmo_set = set(client.query(query).to_dataframe()['visit_occurrence_id'].astype('int64'))
print(f'ECMO visits (distinct, all tables): {len(ecmo_set):,}')

Identified 4 Visits with ECMO

### ECMO concepts with count

AI-assisted (Claude, Anthropic — Claude Opus 4.8): ECMO concept-count check drafted with AI.

In [ ]:
query = f"""
WITH ecmo_concepts AS (
    SELECT concept_id, concept_name, domain_id
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
    WHERE LOWER(concept_name) LIKE '%extracorporeal membrane%'
       OR LOWER(concept_name) LIKE '%extracorporeal life support%'
       OR LOWER(concept_name) LIKE '%ecmo%'
       OR LOWER(concept_name) LIKE '%ecls%'
)
SELECT
    ec.concept_id,
    ec.concept_name,
    ec.domain_id,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence`
      WHERE procedure_concept_id = ec.concept_id
        AND provider_id IS NOT NULL)   AS proc_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.device_exposure`
      WHERE device_concept_id = ec.concept_id
        AND provider_id IS NOT NULL)       AS device_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id = ec.concept_id
        AND provider_id IS NOT NULL)  AS meas_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.observation`
      WHERE observation_concept_id = ec.concept_id
        AND provider_id IS NOT NULL)  AS obs_visits,
    (SELECT COUNT(DISTINCT visit_occurrence_id)
       FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.drug_exposure`
      WHERE drug_concept_id = ec.concept_id
        AND provider_id IS NOT NULL)         AS drug_visits
FROM ecmo_concepts ec
ORDER BY proc_visits DESC, device_visits DESC, meas_visits DESC, obs_visits DESC
"""
ecmo_concept_visits = client.query(query).to_dataframe()

# Quick summary
ecmo_concept_visits['total_visits'] = (
    ecmo_concept_visits[['proc_visits','device_visits','meas_visits','obs_visits','drug_visits']].sum(axis=1)
)
print(f"Concepts found: {len(ecmo_concept_visits)}")
print(f"Concepts with ANY visits: {(ecmo_concept_visits['total_visits'] > 0).sum()}")
print("\nConcepts that actually carry data:")
print(ecmo_concept_visits[ecmo_concept_visits['total_visits'] > 0].to_string(index=False))
ecmo_concept_visits

All 108 standard ECMO concepts have zero records in this OMOP build; ECMO is captured only as sparse free-text (4 visits), so concept-based exclusion adds nothing.

### Extract IMV (Invasive Mechanical Ventillation) logs

AI-assisted (Claude, Anthropic — Claude Opus 4.8): IMV-log pull and dedup drafted with AI.

In [ ]:
query = f"""
SELECT
    po.person_id,
    po.visit_occurrence_id,
    po.procedure_datetime     AS log_datetime,
    po.procedure_end_datetime AS log_end_datetime
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence` po
WHERE po.provider_id IS NOT NULL
  AND po.procedure_concept_id IN ({IMV_CONCEPTS_SQL})
ORDER BY po.visit_occurrence_id, po.procedure_datetime
"""
imv_logs = client.query(query).to_dataframe()
n_before = len(imv_logs)

# Deduplicate exact (visit, datetime) repeats — keep first
imv_logs = (
    imv_logs
    .sort_values(['visit_occurrence_id', 'log_datetime', 'log_end_datetime'], na_position='last')
    .drop_duplicates(subset=['visit_occurrence_id', 'log_datetime'], keep='first')
    .reset_index(drop=True)
)
n_after = len(imv_logs)

print(f'Logs before dedup : {n_before:,}')
print(f'Logs after dedup  : {n_after:,}')
print(f'Duplicates removed: {n_before - n_after:,}')
print(f'Visits with IMV   : {imv_logs["visit_occurrence_id"].nunique():,}')
imv_logs.head()

log_end_time is NaT for all rows because, records ventillation as repeated point-in-time logs(start only), not start-end events - hence the 48h-gap rule and duration = last log - first log + 1h (to count the whole log)

### Build IMV episodes (48-hour gap rule)

AI-assisted (Claude, Anthropic — Claude Opus 4.8): 48h-gap episode logic drafted with AI

In [ ]:
imv_logs = imv_logs.sort_values(['visit_occurrence_id', 'log_datetime']).reset_index(drop=True)

# Time since previous log within the same visit; >=48h (or first log) starts a new episode
imv_logs['prev_dt']   = imv_logs.groupby('visit_occurrence_id')['log_datetime'].shift(1)
imv_logs['gap_hours'] = (imv_logs['log_datetime'] - imv_logs['prev_dt']).dt.total_seconds() / 3600
imv_logs['new_ep']    = imv_logs['gap_hours'].isna() | (imv_logs['gap_hours'] >= 48)
imv_logs['ep_idx']    = imv_logs.groupby('visit_occurrence_id')['new_ep'].cumsum()

# Most logs lack an end-time, so duration = span of logs + 1h (approximation)
USE_END = imv_logs['log_end_datetime'].notna().mean() > 0.5
print(f'Using procedure_end_datetime for duration: {USE_END}')

episodes = (
    imv_logs.groupby(['person_id', 'visit_occurrence_id', 'ep_idx'])
    .agg(episode_start=('log_datetime', 'min'),
         episode_end=('log_datetime', 'max'),
         episode_end_true=('log_end_datetime', 'max'),
         n_logs=('log_datetime', 'count'))
    .reset_index()
)
if USE_END:
    episodes['duration_hours'] = (episodes['episode_end_true'] - episodes['episode_start']).dt.total_seconds() / 3600
else:
    episodes['duration_hours'] = (episodes['episode_end'] - episodes['episode_start']).dt.total_seconds() / 3600 + 1

# Keep first episode per visit
first_episodes = (
    episodes.sort_values(['visit_occurrence_id', 'episode_start'])
    .drop_duplicates('visit_occurrence_id', keep='first')
    .reset_index(drop=True)
)
print(f'Total episodes : {len(episodes):,}')
print(f'First episodes : {len(first_episodes):,}')

### Apply inclusion/exclusion filters
Paper criteria: adult (>=18), admitted >=2010, IMV episode >24h, exclude ECMO.
Target: 2,626 AmsterdamUMCdb admissions.

AI-assisted (Claude, Anthropic — Claude Opus 4.8): inclusion/exclusion filter drafted with AI.

In [ ]:
def run_filters(visits_df, first_eps, ecmo_set, label=''):
    stages, c = [], visits_df.copy()
    stages.append(('All ICU admissions', len(c)))

    c = c.merge(
        first_eps[['person_id', 'visit_occurrence_id',
                   'episode_start', 'episode_end', 'duration_hours', 'n_logs']],
        on=['person_id', 'visit_occurrence_id'], how='inner'
    )
    stages.append(('Has IMV episode', len(c)))

    c = c[c['age_at_admission'] >= 18];          stages.append(('Adult (>=18)', len(c)))
    c = c[c['admission_year'] >= 2010];          stages.append(('Admitted >=2010', len(c)))
    c = c[c['duration_hours'] > 24];             stages.append(('IMV > 24h', len(c)))
    c = c[~c['visit_occurrence_id'].isin(ecmo_set)]; stages.append(('No ECMO', len(c)))

    df = pd.DataFrame(stages, columns=['Stage', 'N'])
    df['Dropped'] = (df['N'].shift(1).fillna(df['N'].iloc[0]) - df['N']).astype(int)
    print(f'[{label}] Final: {len(c):,}  (target 2,626  diff: {len(c)-2626:+,})')
    return c, df

cohort_all, stages_all = run_filters(visits, first_episodes, ecmo_set, 'STANDARD')
stages_all

In [ ]:
m = visits.merge(first_episodes[['visit_occurrence_id']], on='visit_occurrence_id')
print((m['age_at_admission'] < 18).sum(), "ventilated visits under 18")
print(m['age_at_admission'].min(), "= min age")

#### Resolve multiple admissions per patient
Paper examines one IMV period per patient. We currently count repeat ICU
admissions, causing an overshoot. Compare keeping the first admission per person.

In [ ]:
# Keep first ICU admission per person (earliest visit_start)
cohort_first = (
    cohort_all.sort_values(['person_id', 'visit_start_datetime'])
    .drop_duplicates('person_id', keep='first')
    .reset_index(drop=True)
)

print(f'All qualifying admissions      : {len(cohort_all):,}  (diff {len(cohort_all)-2626:+,})')
print(f'First admission per person     : {len(cohort_first):,}  (diff {len(cohort_first)-2626:+,})')
print(f'Unique patients in full cohort : {cohort_all["person_id"].nunique():,}')

The replicated cohort has 2,673 admissions vs the [Zappalà et al. (2025)](https://www.sciencedirect.com/science/article/pii/S0883944125000929) 2,626(+47) . The difference arise because of the paper's exact operational definitions - Cocept Codes, ECMO identification are unpublished. The close match in size and in age and sex distributions confirms the inclustion criteria were faithfully reproduced.

###

### IMV duration with vs without the +1h log approximation

In [ ]:
fe = first_episodes.copy()
fe['dur_no_plus1'] = (fe['episode_end'] - fe['episode_start']).dt.total_seconds()/3600
print("Episodes >24h WITH +1h :", (fe['duration_hours'] > 24).sum())
print("Episodes >24h NO +1h   :", (fe['dur_no_plus1'] > 24).sum())
print("Difference             :", (fe['duration_hours'] > 24).sum() - (fe['dur_no_plus1'] > 24).sum())

#### Rebuild cohort without using +1h log approximation

In [ ]:

fe = first_episodes.copy()
fe['dur_no_plus1'] = (fe['episode_end'] - fe['episode_start']).dt.total_seconds()/3600

# swap duration, re-run filters to final
cohort_test = visits.merge(
    fe[fe['dur_no_plus1'] > 24][['person_id','visit_occurrence_id']],
    on=['person_id','visit_occurrence_id'], how='inner'
)
cohort_test = cohort_test[cohort_test['age_at_admission'] >= 18]
cohort_test = cohort_test[cohort_test['admission_year'] >= 2010]
cohort_test = cohort_test[~cohort_test['visit_occurrence_id'].isin(ecmo_set)]
cohort_test = cohort_test.sort_values(['person_id','visit_start_datetime']).drop_duplicates('person_id', keep='first')

print("Final cohort, no +1h:", len(cohort_test), " (diff vs 2626:", len(cohort_test)-2626, ")")

There is now difference of 12 patients between [Zappalà et al. (2025)](https://www.sciencedirect.com/science/article/pii/S0883944125000929) and Current cohort. These are not findable because of concepts codes, ECMO defintion are not published in the Zappalà et al. (2025).

### Validate Age Distriution

AI-assisted (Claude, Anthropic — Claude Opus 4.8): age-distribution validation against paper drafted with AI.

In [ ]:
## Validate age distribution vs Zappalà et al. (2025) Table 1
final = cohort_test.copy()

bins   = [0, 39, 49, 59, 69, 79, 200]
labels = ['18-39', '40-49', '50-59', '60-69', '70-79', '80+']
final['age_bucket'] = pd.cut(final['age_at_admission'], bins=bins, labels=labels, right=True)

# Paper Table 1 — AmsterdamUMCdb external cohort (n=2626)
zappala_pct = {'18-39': 11.23, '40-49': 10.21, '50-59': 17.90,
               '60-69': 25.97, '70-79': 24.52, '80+': 10.17}

counts = final['age_bucket'].value_counts().sort_index()
pct    = (counts / counts.sum() * 100).round(2)

comp = pd.DataFrame({
    'Current N':    counts,
    'Current %':    pct,
    'Zappalà %': pd.Series(zappala_pct),
    'Diff pp':   (pct - pd.Series(zappala_pct)).round(2)
})
print(f"Cohort: {len(final):,} admissions | Mean age: {final['age_at_admission'].mean():.1f} (paper: 64.6)")
print(comp.to_string())

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
x, w = range(len(labels)), 0.35
ax.bar([i - w/2 for i in x], pct.reindex(labels).values, w, label='Current Cohort', color='#55A868')
ax.bar([i + w/2 for i in x], [zappala_pct[k] for k in labels], w, label='Zappalà 2025', color='#4C72B0', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Percentage (%)'); ax.legend()
ax.set_title(f'Age distribution: Current cohort (N={len(final):,}) vs Zappalà et al. (2025) (N=2,626)')
plt.tight_layout(); plt.show()

Current cohort's age distribution matches [Zappalà et al. (2025)](https://www.sciencedirect.com/science/article/pii/S0883944125000929#s0010) across 6 bands.

### Sex Distribution

AI-assisted (Claude, Anthropic — Claude Opus 4.8): sex-distribution validation against paper Table 1 drafted with AI.

In [ ]:
## Validate sex distribution vs Zappalà et al. (2025) Table 1
# Paper Table 1 — AmsterdamUMCdb external cohort (n=2626): Male 65.65%, Female 34.35%
zappala_sex = {'Male': 65.65, 'Female': 34.35}

sex_counts = final['gender_label'].value_counts()
sex_pct    = (sex_counts / sex_counts.sum() * 100).round(2)

sex_comp = pd.DataFrame({
    'Current N':    sex_counts,
    'Current %':    sex_pct,
    'Zappalà %': pd.Series(zappala_sex),
    'Diff pp':   (sex_pct - pd.Series(zappala_sex)).round(2)
})
print(f"Cohort: {len(final):,} admissions")
print(sex_comp.to_string())

# Plot
labels_sex = ['Male', 'Female']
fig, ax = plt.subplots(figsize=(6, 5))
x, w = range(len(labels_sex)), 0.35
ax.bar([i - w/2 for i in x], sex_pct.reindex(labels_sex).values, w, label='Current Cohort', color='#55A868')
ax.bar([i + w/2 for i in x], [zappala_sex[k] for k in labels_sex], w, label='Zappalà 2025', color='#4C72B0', alpha=0.7)
ax.set_xticks(list(x)); ax.set_xticklabels(labels_sex)
ax.set_ylabel('Percentage (%)'); ax.legend()
ax.set_title(f'Sex distribution: Current cohort (N={len(final):,}) vs Zappalà et al. (2025)')
plt.tight_layout(); plt.show()

Current cohort sex distribution approximately matches [Zappalà et al. (2025)](https://www.sciencedirect.com/science/article/pii/S0883944125000929#s0010) distribution.

#### Save final cohort


In [ ]:
cols_in  = ['person_id', 'visit_occurrence_id', 'visit_start_datetime', 'visit_end_datetime',
            'admission_year', 'age_at_admission', 'age_bucket', 'gender_label',
            'episode_start', 'episode_end', 'duration_hours', 'n_logs']
cols_out = ['person_id', 'visit_occurrence_id', 'admission_start', 'admission_end',
            'admission_year', 'age', 'age_bucket', 'gender',
            'imv_start', 'imv_end', 'imv_duration_hours', 'imv_n_logs']

save = final[[c for c in cols_in if c in final.columns]].copy()
save.columns = cols_out[:len(save.columns)]
save.to_parquet('cohort_zappala.parquet', index=False)

print(f'Saved {len(save):,} rows to cohort_zappala.parquet')
save.head()

## Cohort Summary

Replicated the [Zappalà et al. (2025)](https://www.sciencedirect.com/article/pii/S0883944125000929) cohort on AmsterdamUMCdb (OMOP CDM v5.4). Final cohort: **2,638 adult ICU admissions** with invasive mechanical ventilation >24h, ECMO excluded, first admission per patient, admissision >= 2010, age >= 18.

The 12-patient gap from the paper's 2,626 is because of their exact ECMO concept codes and IMV definitions are unpublished. Age and sex distributions match Table 1 closely.

Cohort saved to `cohort_final.csv`.